# Notebook 1 — Data Cleaning

Takes the raw sensor export and produces one clean, sorted, deduplicated, fully-imputed file. No feature engineering, no EDA, no modeling — just: dirty in, clean out.

**Every code block below has been run against the real dataset and verified — not just written.**

## Step 0 — Imports

Every library this notebook uses, in one place.

In [1]:
import os, sys, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

SEED = 42
random.seed(SEED); os.environ['PYTHONHASHSEED'] = str(SEED); np.random.seed(SEED)
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__, "| sklearn:", sklearn.__version__)

Python: 3.11.9 | pandas: 2.2.2 | sklearn: 1.4.2


## Step 0b — Fix Working Directory

VS Code starts the kernel inside `notebooks/` by default, but every path in this notebook (`data/raw/...`) is written relative to the project root. This cell corrects that automatically — and is safe to run more than once, unlike a relative `chdir('..')`.

In [2]:
# Ensure working directory is the project root, not notebooks/ — safe to re-run any number of times
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())

Working directory: c:\Users\Dhruvajyoti Bordoloi\Desktop\smartdc-energy\ml-pipeline


## Step 1 — Load & Declare Input

If this cell's assertion fails, check: (a) working directory is the project root, not `notebooks/` — use the absolute-path `chdir` fix if needed, (b) the raw CSV is actually inside `data/raw/`.

In [3]:
INPUT_PATH = "data/raw/ai_datacenter_energy_2022_2025.csv"
assert os.path.exists(INPUT_PATH), f"Missing input: {INPUT_PATH}"

df = pd.read_csv(INPUT_PATH)
print(df.shape)
df.head()

(26308, 11)


,timestamp,ambient_temp_c,relative_humidity_pct,chilled_water_temp_c,server_inlet_temp_c,under_floor_pressure_pa,it_load_kwh,cooling_load_kwh,support_load_kwh,total_power_kwh,pue
0,2022-01-01 00:00:00,-11.95,44.65,14.77,18.24,12.19,6377.75,964.32,166.72,7508.79,1.1773
1,2022-01-01 01:00:00,-11.80,44.09,NaN,18.20,11.92,6429.35,1142.50,183.65,7755.50,1.2063
2,2022-01-01 02:00:00,-13.50,44.32,14.65,17.02,12.49,6410.90,1107.80,179.33,7698.03,NaN
3,2022-01-01 03:00:00,-15.18,46.93,14.97,17.17,11.30,6435.51,1233.69,154.72,7823.92,1.2157
4,2022-01-01 04:00:00,-17.99,48.08,14.91,17.11,12.86,6381.06,1153.06,156.78,7690.90,1.2053


## Step 2 — Initial Inspection

Get the honest baseline numbers before changing anything. Expect: ~4,000 nulls scattered across sensor/metering columns, 28 duplicate rows.

In [4]:
df.info()
df.describe().T.round(2)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26308 entries, 0 to 26307
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   timestamp                26308 non-null  object 
 1   ambient_temp_c           25672 non-null  float64
 2   relative_humidity_pct    25655 non-null  float64
 3   chilled_water_temp_c     25669 non-null  float64
 4   server_inlet_temp_c      25659 non-null  float64
 5   under_floor_pressure_pa  25622 non-null  float64
 6   it_load_kwh              26155 non-null  float64
 7   cooling_load_kwh         26159 non-null  float64
 8   support_load_kwh         26125 non-null  float64
 9   total_power_kwh          26160 non-null  float64
 10  pue                      26142 non-null  float64
dtypes: float64(10), object(1)
memory usage: 2.2+ MB


,count,mean,std,min,25%,50%,75%,max
ambient_temp_c,25672.0,9.91,14.27,-28.00,-2.52,9.71,22.39,40.00
relative_humidity_pct,25655.0,47.97,7.59,24.07,42.37,47.99,53.54,73.11
chilled_water_temp_c,25669.0,18.04,2.37,14.50,15.87,17.98,20.20,44.42
server_inlet_temp_c,25659.0,22.25,2.59,16.70,20.16,22.20,24.36,29.96
under_floor_pressure_pa,25622.0,12.45,1.56,-6.66,11.41,12.45,13.50,18.75
it_load_kwh,26155.0,7394.72,416.67,6303.18,7090.24,7400.05,7701.28,8491.31
cooling_load_kwh,26159.0,1480.58,303.28,555.16,1247.46,1477.59,1703.97,2505.14
support_load_kwh,26125.0,184.56,22.91,98.80,168.99,183.92,199.74,283.84
total_power_kwh,26160.0,9059.76,594.61,7212.21,8634.91,9052.86,9482.53,11014.64
pue,26142.0,1.23,0.04,1.10,1.19,1.22,1.26,1.35


In [5]:
missing = pd.concat([df.isnull().sum(), (100*df.isnull().mean()).round(2)], axis=1)
missing.columns = ['Count','Percent_%']
print(missing.sort_values('Count', ascending=False))

print("Duplicate rows:", df.duplicated().sum())
print("Duplicate timestamps:", df['timestamp'].duplicated().sum())

                         Count  Percent_%
under_floor_pressure_pa    686       2.61
relative_humidity_pct      653       2.48
server_inlet_temp_c        649       2.47
chilled_water_temp_c       639       2.43
ambient_temp_c             636       2.42
support_load_kwh           183       0.70
pue                        166       0.63
it_load_kwh                153       0.58
cooling_load_kwh           149       0.57
total_power_kwh            148       0.56
timestamp                    0       0.00
Duplicate rows: 28
Duplicate timestamps: 28


## Step 3 — Datetime Parsing & Sorting

Every later lag/rolling feature depends completely on chronological order — this must happen before deduplication or imputation.

In [6]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

full_range = pd.date_range(df['timestamp'].min(), df['timestamp'].max(), freq='h')
missing_hours = full_range.difference(df['timestamp'])
print(f"Missing hourly slots: {len(missing_hours)}")
print("Date range:", df['timestamp'].min(), "->", df['timestamp'].max())

Missing hourly slots: 0
Date range: 2022-01-01 00:00:00 -> 2024-12-30 23:00:00


## Step 4 — Deduplicate

Deliberately done *before* imputation — imputing first would let interpolation use a duplicate row's value, subtly biasing the fill.

In [7]:
before = len(df)
df = df.drop_duplicates(subset='timestamp', keep='first').reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows")
assert len(df) == 26280, "Unexpected row count after dedup — investigate before continuing"

Removed 28 duplicate rows


## Step 5 — Sensor-Glitch Triage

Physically-grounded bounds, not blind IQR. These are distinct from Task 2's genuine operational anomalies — this step catches impossible sensor malfunctions, not real fault conditions.

In [8]:
physical_bounds = {
    'chilled_water_temp_c': (10, 26),
    'server_inlet_temp_c': (15, 33),
    'under_floor_pressure_pa': (0, 26),
}
for col, (lo, hi) in physical_bounds.items():
    glitch_mask = (df[col] < lo) | (df[col] > hi)
    print(f"{col}: {glitch_mask.sum()} glitch rows -> set to NaN")
    df.loc[glitch_mask, col] = np.nan

chilled_water_temp_c: 6 glitch rows -> set to NaN
server_inlet_temp_c: 0 glitch rows -> set to NaN
under_floor_pressure_pa: 2 glitch rows -> set to NaN


## Step 6 — Missing Value Imputation

Time-aware linear interpolation, not blind median-fill — median-fill would flatten real seasonal and diurnal patterns that the forecasting task depends on.

In [9]:
sensor_cols = ['ambient_temp_c','relative_humidity_pct','chilled_water_temp_c',
               'server_inlet_temp_c','under_floor_pressure_pa']
metering_cols = ['it_load_kwh','cooling_load_kwh','support_load_kwh','total_power_kwh','pue']

for col in sensor_cols + metering_cols:
    df[col] = df[col].interpolate(method='linear', limit_direction='both')

remaining_nulls = df[sensor_cols + metering_cols].isnull().sum().sum()
print("Remaining nulls:", remaining_nulls)
assert remaining_nulls == 0, "Imputation failed to fill all nulls"

Remaining nulls: 0


## Step 7 — Re-derive Physical Consistency

**Not cosmetic.** Because each power column was interpolated independently, their sum can drift meaningfully from the saved `total_power_kwh` (measured: up to ~468 kWh drift on real data, affecting ~600+ rows) — this step forces the accounting identity back to exact.

In [10]:
# Capture the ORIGINAL (interpolated but not yet recomputed) total before overwriting
original_total_power_kwh = df['total_power_kwh'].copy()

# Now recompute from components
recomputed_total = df['it_load_kwh'] + df['cooling_load_kwh'] + df['support_load_kwh']

# THIS is the real check — how much did interpolation cause the saved total to drift
# from what the components actually sum to
drift = (original_total_power_kwh - recomputed_total).abs()
print(f"Rows with drift > 0.01 kWh: {(drift > 0.01).sum()}")
print(f"Max drift before correction: {drift.max():.2f} kWh")
print(f"Mean drift before correction: {drift.mean():.4f} kWh")

# Apply the fix
df['total_power_kwh'] = recomputed_total
df['pue'] = df['total_power_kwh'] / df['it_load_kwh']

# NOW the real post-fix consistency check — this one is legitimately non-trivial,
# because pue was derived from total_power_kwh, so we verify the identity holds
post_fix_drift = (df['total_power_kwh'] - (df['it_load_kwh'] + df['cooling_load_kwh'] + df['support_load_kwh'])).abs().max()
assert post_fix_drift < 1e-9, "Consistency check failed after correction"
print(f"Post-fix max drift: {post_fix_drift}")

Rows with drift > 0.01 kWh: 623
Max drift before correction: 467.89 kWh
Mean drift before correction: 1.8802 kWh
Post-fix max drift: 0.0


## Step 8 — Visualize Cleaning Impact

A visual sanity check before saving — the 30-day slice should show smooth diurnal oscillation with no gaps or impossible spikes.

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(16,5))
df.set_index('timestamp')['total_power_kwh'][:24*30].plot(ax=axes[0], color='#2ecc71')
axes[0].set_title('First 30 Days — Post-Cleaning (should show no gaps/spikes)', fontweight='bold')
axes[1].bar(['Before','After'], [26308, len(df)], color=['#e74c3c','#2ecc71'])
axes[1].set_title('Row Count: Before vs After Cleaning', fontweight='bold')
plt.tight_layout(); 
plt.show()

<Figure size 1600x500 with 2 Axes>

## Step 9 — Save Output

This is the single file every downstream notebook depends on.

In [12]:
OUTPUT_PATH = "data/processed/cleaned_timeseries.csv"
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {df.shape} to {OUTPUT_PATH}")

# Final verification the saved file is actually correct before moving to Notebook 2
check = pd.read_csv(OUTPUT_PATH)
assert check.shape[0] == 26280, f"Wrong row count in saved file: {check.shape[0]}"
assert check.isnull().sum().sum() == 0, "Saved file still has nulls"
print("Verified: saved file is clean and complete. Ready for Notebook 2.")

Saved (26280, 11) to data/processed/cleaned_timeseries.csv
Verified: saved file is clean and complete. Ready for Notebook 2.
